# Video Preprocess Demo

이 notebook 은 `video-preprocess-workbench` 를 Jupyter 에서 사용하는 상세 예제다.

권장 순서:
1. 프로젝트 루트와 `src/` import 준비
2. `example_batch.json` 을 로드하고 내 PC 경로로 override
3. `inspect_inputs()` 로 영상 개수, 해상도, FPS 분포를 먼저 확인
4. `save_preview()` 로 대표 프레임 preview 와 ROI overlay 확인
5. `run_batch(..., dry_run=True)` 로 실제 인코딩 없이 보고서만 확인
6. 문제가 없으면 `run_batch(..., dry_run=False)` 로 실제 실행

실무 팁:
- 처음에는 `scan_depth=0` 또는 `limit=3` 정도로 작게 확인하는 것이 안전하다.
- 세로 영상이나 해상도가 제각각인 데이터셋은 `resize_mode='fit_pad'` 를 먼저 추천한다.
- ROI blur 를 쓸 때는 `preview/frame_with_rois.jpg` 를 꼭 확인한다.


In [ ]:
from pathlib import Path
import sys

# notebook 실행 위치가 repo 루트이든 notebooks 폴더이든 상관없이
# pyproject.toml 과 src 폴더를 기준으로 프로젝트 루트를 찾는다.
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    raise RuntimeError('project root not found')

project_root = find_project_root(Path.cwd().resolve())
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from video_preprocess_workbench import load_config, inspect_inputs, save_preview, run_batch
from video_preprocess_workbench.pipeline import apply_overrides

print('project_root =', project_root)
print('src_dir =', src_dir)


## 설정 로드 및 override

아래 셀에서 가장 자주 바꾸는 값은 다음 4개다.

- `input_path`: 내 원본 영상 폴더 또는 파일
- `output_dir`: 결과를 저장할 폴더
- `scan_depth`: `0`, `1`, `2` 또는 전체 재귀면 `None`
- `target_fps`: 출력 FPS

자주 쓰는 패턴:
- 폴더 바로 아래 영상만 처리: `scan_depth=0`
- 하위 폴더 포함 전체 처리: `scan_depth=None`


In [ ]:
config_path = project_root / 'configs' / 'example_batch.json'
cfg = load_config(config_path)

# input_path 는 사용자가 실제로 바꿔야 하는 핵심 값이다.
# scan_depth=0 이면 입력 폴더 바로 아래 파일만 처리한다.
# scan_depth=None 으로 바꾸면 전체 하위 폴더를 재귀 스캔한다.
cfg = apply_overrides(
    cfg,
    input_path='/share_ssd/ltb/Users/ltb/박스_추론용_샘플영상들/260413_배테스트용_영상들',
    output_dir=str(project_root / 'artifacts' / 'runs'),
    scan_depth=0,
)

# resize_mode='fit_pad' 는 원본 비율을 유지하고 남는 공간을 padding 한다.
# 여러 해상도/세로영상이 섞인 경우 가장 먼저 시도할 안전한 기본값이다.
cfg.transform.resize_mode = 'fit_pad'
cfg.transform.resize_width = 1280
cfg.transform.resize_height = 720

# target_fps 는 출력 FPS 목표값이다.
# fps_mode='downsample_only' 이면 원본 FPS가 더 낮을 때 억지로 올리지 않는다.
cfg.transform.target_fps = 10.0

# ROI blur 를 쓰지 않을 때는 False 로 둔다.
# 쓰고 싶으면 아래 값을 True 로 바꾸고, normalized box 를 넣는다.
cfg.roi.enabled = False

# preview 에서 몇 번째 영상을 샘플로 볼지 지정한다.
cfg.preview.video_index = 0
cfg.preview.debug_frame_index = 0

cfg.to_dict()


## 1차 점검: inventory

이 단계에서는 실제 변환을 하지 않는다.

확인할 것:
- `total_files`
- `failed_files`
- 해상도 분포 `resolutions`
- FPS 분포 `fps_values`

대형 폴더라면 `inspect_inputs(cfg, limit=5)` 처럼 일부만 먼저 볼 수 있다.


In [ ]:
inspection = inspect_inputs(cfg)
inspection['summary']


## 2차 점검: preview

이 단계에서는 대표 프레임 1장을 변환해서 아래 파일을 저장한다.

- `frame_transformed.jpg`: resize/crop 결과
- `frame_with_grid.jpg`: 좌표 보기 쉽게 grid 추가
- `frame_with_rois.jpg`: ROI overlay 추가

ROI blur 를 쓸 계획이면 반드시 이 단계에서 좌표를 확인하는 것을 권장한다.


In [ ]:
preview = save_preview(cfg, run_dir=inspection['run_dir'])
preview['preview_info']


## 3차 점검: dry-run

`dry_run=True` 는 실제 인코딩 없이 report/preview 흐름만 검증한다.
새 폴더 구조나 scan depth 가 맞는지 확인할 때 먼저 사용하는 것을 추천한다.


In [ ]:
summary = run_batch(cfg, dry_run=True, run_dir=inspection['run_dir'])
summary


## 실제 실행 예시

아래 코드는 기본적으로 주석 처리해 두는 것을 권장한다.
preview 와 dry-run 결과가 괜찮을 때만 실행한다.


In [ ]:
# real_summary = run_batch(cfg, dry_run=False, run_dir=inspection['run_dir'])
# real_summary
